# Experiment 2 — Active bomb-placement escape redundancy (`e2`)

**Research question.** Does requiring two independent first-step bomb-escape branches reduce catastrophic self-traps enough to justify potentially fewer productive bombs?

**Additional framing.** Before placing a bomb, does requiring at least two independent first-step escape branches substantially reduce the risk of becoming trapped after placement, and is that safety benefit worth the potential loss in bombs, crates, or total score?

This is not an experiment on the passive one-step lethality mask. The shared immediate-lethality filter runs first. The three arms require at least two branches for every bomb, use the original hybrid rule, or permit every one-branch bomb, respectively.

## Locked protocol

- Candidate/checkpoint: `dueling_hunt_center_04` / `709cca4351f3bc5893f766b01b8c9335fa299d25331a275ae281b123df31009a`
- Conditions: `two_branch_final` (≥2), `original_hybrid` (original quiet-opening exception), `one_branch_relaxed` (≥1)
- Core lineups: `external_mixed`, `frozen_dddqn`
- Fresh seeds: `66001–66040`; 5 rounds per seed
- Paired unit: seed cluster after averaging its five rounds
- Total: 1200 rounds; zero exploration; 10,000 paired bootstrap resamples for every pair
- Locked: checkpoint, danger model, objective routing, hunt-centre bias λ=.04, opponent lineup, engine files and code hashes

Implementation note: the repository feature extractor's quiet-opening one-branch exception is now a third, explicitly measured arm. The original two arms remain unchanged append-only records. All three leave the checkpoint, feature values, adjusted Q-values, passive survival filter, round seeding and opponents unchanged at a shared state.

Raw stats, decision traces, bomb-event traces, manifests, SHA-256 ledger and seed-level records are under `results/e2_bomb_escape_branches/`.

In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path('..').resolve()
payload = json.loads((ROOT / 'results/e2_bomb_escape_branches/summary_threeway.json').read_text())
aggregate = pd.DataFrame(payload['aggregate'])
seed_level = pd.DataFrame(payload['seed_rows'])
paired = pd.DataFrame(payload['paired_bootstrap'])

## Pairing and completeness

Every condition has 40 complete seed clusters × 5 rounds. The table counts exact initial-condition hash matches across all three arms; 200/200 is required for each lineup.

| lineup | matched | expected |
| --- | --- | --- |
| external_mixed | 200 | 200 |
| frozen_dddqn | 200 | 200 |

## Aggregate results and exposure integrity check

`score`, `bombs`, `crates`, `coins`, `suicides`, and `survived` are per round. `decision_ms_p95` pools decision-level timings. Counts ending in `_total` are totals over 200 rounds per row. `self_traps_per_100_bombs` is a descriptive exposure-normalized rate.

| lineup | variant | rounds | score | bombs | crates | coins | suicides | survived | self_trapped_total | bomb_proposals_total | bomb_rejections_total | actual_bombs_total | self_traps_per_100_bombs | decision_ms_p95 |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| external_mixed | two_branch_final | 200 | 1.3200 | 25.8350 | 5.8750 | 0.5700 | 0.1750 | 0.7700 | 32 | 26347 | 21180 | 5167 | 0.6193 | 9.8809 |
| external_mixed | original_hybrid | 200 | 2.5050 | 41.5150 | 23.4150 | 1.7550 | 0.1750 | 0.7200 | 35 | 11847 | 3544 | 8303 | 0.4215 | 12.3615 |
| external_mixed | one_branch_relaxed | 200 | 2.9850 | 45.9350 | 27.7950 | 2.0600 | 0.2100 | 0.7550 | 52 | 9188 | 0 | 9187 | 0.5660 | 12.9539 |
| frozen_dddqn | two_branch_final | 200 | 0.9300 | 20.4550 | 4.1050 | 0.4050 | 0.1700 | 0.7650 | 33 | 31234 | 27143 | 4091 | 0.8066 | 12.8350 |
| frozen_dddqn | original_hybrid | 200 | 3.0900 | 39.9150 | 28.3050 | 1.9650 | 0.2250 | 0.6450 | 62 | 12329 | 4346 | 7983 | 0.7767 | 14.7326 |
| frozen_dddqn | one_branch_relaxed | 200 | 3.8950 | 44.5350 | 32.9200 | 2.2450 | 0.2600 | 0.6650 | 102 | 8907 | 0 | 8907 | 1.1452 | 14.0371 |

## Paired seed-cluster bootstrap

All three pairwise differences are reported. For each `left-right` contrast, positive values favor the left arm for score/productivity/survival; negative values favor it for suicides, self-traps, and rejections. Intervals are percentile 95% paired-bootstrap intervals over the 40 seed means.

| lineup | contrast | metric | mean_difference | ci_low | ci_high | positive_seeds | zero_seeds | negative_seeds |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| external_mixed | two_branch_final-one_branch_relaxed | score | -1.6650 | -2.0850 | -1.2400 | 3 | 1 | 36 |
| external_mixed | two_branch_final-one_branch_relaxed | coins | -1.4900 | -1.6950 | -1.2850 | 0 | 0 | 40 |
| external_mixed | two_branch_final-one_branch_relaxed | crates | -21.9200 | -22.9950 | -20.7600 | 0 | 0 | 40 |
| external_mixed | two_branch_final-one_branch_relaxed | bombs | -20.1000 | -22.8651 | -17.2950 | 0 | 0 | 40 |
| external_mixed | two_branch_final-one_branch_relaxed | suicides | -0.0350 | -0.0900 | 0.0200 | 11 | 14 | 15 |
| external_mixed | two_branch_final-one_branch_relaxed | survived | 0.0150 | -0.0401 | 0.0750 | 11 | 18 | 11 |
| external_mixed | two_branch_final-one_branch_relaxed | bomb_proposals | 85.7950 | 77.3349 | 94.3102 | 40 | 0 | 0 |
| external_mixed | two_branch_final-one_branch_relaxed | bomb_rejections | 105.9000 | 96.0649 | 116.3858 | 40 | 0 | 0 |
| external_mixed | two_branch_final-one_branch_relaxed | self_trapped_after_bomb | -0.1000 | -0.1650 | -0.0350 | 8 | 11 | 21 |
| external_mixed | two_branch_final-original_hybrid | score | -1.1850 | -1.5150 | -0.8600 | 5 | 1 | 34 |
| external_mixed | two_branch_final-original_hybrid | coins | -1.1850 | -1.3500 | -1.0150 | 1 | 0 | 39 |
| external_mixed | two_branch_final-original_hybrid | crates | -17.5400 | -18.7851 | -16.2600 | 0 | 0 | 40 |
| external_mixed | two_branch_final-original_hybrid | bombs | -15.6800 | -19.2100 | -12.2450 | 4 | 0 | 36 |
| external_mixed | two_branch_final-original_hybrid | suicides | -0.0000 | -0.0700 | 0.0750 | 14 | 12 | 14 |
| external_mixed | two_branch_final-original_hybrid | survived | 0.0500 | -0.0450 | 0.1450 | 18 | 8 | 14 |
| external_mixed | two_branch_final-original_hybrid | bomb_proposals | 72.5000 | 63.7997 | 81.4250 | 40 | 0 | 0 |
| external_mixed | two_branch_final-original_hybrid | bomb_rejections | 88.1800 | 78.5400 | 97.8801 | 40 | 0 | 0 |
| external_mixed | two_branch_final-original_hybrid | self_trapped_after_bomb | -0.0150 | -0.0900 | 0.0600 | 13 | 13 | 14 |
| external_mixed | original_hybrid-one_branch_relaxed | score | -0.4800 | -0.8850 | -0.0800 | 11 | 2 | 27 |
| external_mixed | original_hybrid-one_branch_relaxed | coins | -0.3050 | -0.5000 | -0.1150 | 9 | 7 | 24 |
| external_mixed | original_hybrid-one_branch_relaxed | crates | -4.3800 | -5.4150 | -3.3600 | 3 | 0 | 37 |
| external_mixed | original_hybrid-one_branch_relaxed | bombs | -4.4200 | -7.6650 | -1.1950 | 14 | 0 | 26 |
| external_mixed | original_hybrid-one_branch_relaxed | suicides | -0.0350 | -0.1100 | 0.0350 | 13 | 11 | 16 |
| external_mixed | original_hybrid-one_branch_relaxed | survived | -0.0350 | -0.1250 | 0.0600 | 14 | 7 | 19 |
| external_mixed | original_hybrid-one_branch_relaxed | bomb_proposals | 13.2950 | 9.5250 | 16.8450 | 33 | 0 | 7 |
| external_mixed | original_hybrid-one_branch_relaxed | bomb_rejections | 17.7200 | 16.0250 | 19.4000 | 40 | 0 | 0 |
| external_mixed | original_hybrid-one_branch_relaxed | self_trapped_after_bomb | -0.0850 | -0.1600 | -0.0100 | 9 | 14 | 17 |
| frozen_dddqn | two_branch_final-one_branch_relaxed | score | -2.9650 | -3.5051 | -2.4150 | 2 | 0 | 38 |
| frozen_dddqn | two_branch_final-one_branch_relaxed | coins | -1.8400 | -2.0150 | -1.6650 | 0 | 0 | 40 |
| frozen_dddqn | two_branch_final-one_branch_relaxed | crates | -28.8150 | -30.0800 | -27.5750 | 0 | 0 | 40 |
| frozen_dddqn | two_branch_final-one_branch_relaxed | bombs | -24.0800 | -26.7500 | -21.4398 | 0 | 0 | 40 |
| frozen_dddqn | two_branch_final-one_branch_relaxed | suicides | -0.0900 | -0.1650 | -0.0150 | 8 | 14 | 18 |
| frozen_dddqn | two_branch_final-one_branch_relaxed | survived | 0.1000 | 0.0150 | 0.1850 | 21 | 10 | 9 |
| frozen_dddqn | two_branch_final-one_branch_relaxed | bomb_proposals | 111.6350 | 104.8100 | 118.5352 | 40 | 0 | 0 |
| frozen_dddqn | two_branch_final-one_branch_relaxed | bomb_rejections | 135.7150 | 128.1250 | 143.3351 | 40 | 0 | 0 |
| frozen_dddqn | two_branch_final-one_branch_relaxed | self_trapped_after_bomb | -0.3450 | -0.4400 | -0.2500 | 3 | 6 | 31 |
| frozen_dddqn | two_branch_final-original_hybrid | score | -2.1600 | -2.6100 | -1.7300 | 3 | 0 | 37 |
| frozen_dddqn | two_branch_final-original_hybrid | coins | -1.5600 | -1.7750 | -1.3500 | 0 | 0 | 40 |
| frozen_dddqn | two_branch_final-original_hybrid | crates | -24.2000 | -25.3751 | -23.0350 | 0 | 0 | 40 |
| frozen_dddqn | two_branch_final-original_hybrid | bombs | -19.4600 | -21.8251 | -17.0249 | 1 | 0 | 39 |
| frozen_dddqn | two_branch_final-original_hybrid | suicides | -0.0550 | -0.1200 | 0.0100 | 7 | 17 | 16 |
| frozen_dddqn | two_branch_final-original_hybrid | survived | 0.1200 | 0.0400 | 0.1950 | 21 | 13 | 6 |
| frozen_dddqn | two_branch_final-original_hybrid | bomb_proposals | 94.5250 | 88.3949 | 100.8903 | 40 | 0 | 0 |
| frozen_dddqn | two_branch_final-original_hybrid | bomb_rejections | 113.9850 | 106.5548 | 121.6502 | 40 | 0 | 0 |
| frozen_dddqn | two_branch_final-original_hybrid | self_trapped_after_bomb | -0.1450 | -0.2300 | -0.0650 | 6 | 13 | 21 |
| frozen_dddqn | original_hybrid-one_branch_relaxed | score | -0.8050 | -1.3000 | -0.3300 | 13 | 1 | 26 |
| frozen_dddqn | original_hybrid-one_branch_relaxed | coins | -0.2800 | -0.4450 | -0.1100 | 8 | 5 | 27 |
| frozen_dddqn | original_hybrid-one_branch_relaxed | crates | -4.6150 | -5.7251 | -3.4700 | 2 | 0 | 38 |
| frozen_dddqn | original_hybrid-one_branch_relaxed | bombs | -4.6200 | -6.9450 | -2.3100 | 11 | 0 | 29 |
| frozen_dddqn | original_hybrid-one_branch_relaxed | suicides | -0.0350 | -0.1200 | 0.0450 | 12 | 14 | 14 |
| frozen_dddqn | original_hybrid-one_branch_relaxed | survived | -0.0200 | -0.1050 | 0.0700 | 14 | 11 | 15 |
| frozen_dddqn | original_hybrid-one_branch_relaxed | bomb_proposals | 17.1100 | 13.9799 | 20.2500 | 36 | 2 | 2 |
| frozen_dddqn | original_hybrid-one_branch_relaxed | bomb_rejections | 21.7300 | 20.1649 | 23.2900 | 40 | 0 | 0 |
| frozen_dddqn | original_hybrid-one_branch_relaxed | self_trapped_after_bomb | -0.2000 | -0.3000 | -0.0950 | 9 | 4 | 27 |

### Exposure-normalized safety diagnostic

| lineup | contrast | metric | mean_difference | ci_low | ci_high | positive_seeds | zero_seeds | negative_seeds |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| external_mixed | two_branch_final-one_branch_relaxed | self_traps_per_100_actual_bombs | 0.0684 | -0.1633 | 0.3063 | 20 | 1 | 19 |
| external_mixed | two_branch_final-original_hybrid | self_traps_per_100_actual_bombs | 0.1953 | -0.0711 | 0.4722 | 19 | 7 | 14 |
| external_mixed | original_hybrid-one_branch_relaxed | self_traps_per_100_actual_bombs | -0.1269 | -0.3472 | 0.0921 | 17 | 3 | 20 |
| frozen_dddqn | two_branch_final-one_branch_relaxed | self_traps_per_100_actual_bombs | -0.3640 | -0.6868 | -0.0294 | 13 | 1 | 26 |
| frozen_dddqn | two_branch_final-original_hybrid | self_traps_per_100_actual_bombs | 0.0187 | -0.2946 | 0.3325 | 17 | 5 | 18 |
| frozen_dddqn | original_hybrid-one_branch_relaxed | self_traps_per_100_actual_bombs | -0.3827 | -0.6366 | -0.1247 | 12 | 0 | 28 |

## Operational trace definitions

- `bomb_proposal`: `BOMB` is among the maximum adjusted-Q actions after the shared immediate-lethality filter, before applying branch redundancy.
- `bomb_rejected`: a bomb proposal fails the arm's branch threshold.
- `self_trapped_after_bomb`: after a real bomb placement and before that bomb explodes, at least one observed post-action world state has no legal action with a survivable continuation under the locked danger simulator.
- Different continuations sharing the same first move count as one branch, not multiple branches.

This attribution is narrower than total suicides: an avoidable bad choice despite an available path is not relabelled as a forced self-trap.

## Decision

**Conclusion: the original hybrid rule clearly repairs the main productivity collapse caused by strict two-branch filtering.** The strict arm's additional safety benefit over the hybrid is not measurable in `external_mixed`; although present against `frozen_dddqn`, it comes with very large score and economic losses, so strict two-branch filtering should not be the final rule. The fully relaxed arm has clearly higher mean score, bombs, crates, and coins in both lineups, whereas the hybrid clearly reduces actual self-traps per round in both. Total suicides and survival do not show an equally clear hybrid advantage. The two arms therefore present a genuine productivity–direct-safety trade-off: choose relaxed if the official mean score is the priority; choose hybrid as a defensible compromise if a limited productivity cost is acceptable in exchange for fewer direct self-traps.

## Reproduction

```bash
/opt/homebrew/Caskroom/miniconda/base/envs/ml_essential/bin/python evaluation/e2_bomb_escape_branches.py status
/opt/homebrew/Caskroom/miniconda/base/envs/ml_essential/bin/python evaluation/e2_hybrid_extension.py status
/opt/homebrew/Caskroom/miniconda/base/envs/ml_essential/bin/python evaluation/e2_hybrid_extension.py summarize
```

The locked protocol and every run manifest contain hashes. Completed raw runs are append-only; the summarizer refuses missing clusters, protocol mismatches, or changed result hashes.